# AfriSenti Twi downstream matrix

Recreates the nine selected revision-v2 checkpoints one at a time, evaluates two unadapted bases, and writes resumable downstream result JSONs. Attach `akan-bpe-revision-v2-data` and, after the first session, `akan-bpe-downstream-results`. Use a T4 accelerator with internet enabled.

In [ ]:
import os
from pathlib import Path
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
if Path.cwd().name != 'akan-bpe':
    if not Path('akan-bpe').is_dir():
        !git clone https://github.com/attabeezy/akan-bpe.git
    %cd akan-bpe
print(Path.cwd())

In [ ]:
import subprocess, sys
packages = ['datasets>=2.18.0', 'transformers>=4.51.0', 'tokenizers>=0.15.0', 'scikit-learn>=1.4.0', 'numpy>=1.26.0', 'protobuf>=5.29.0', 'PyYAML>=6.0', 'python-dotenv>=1.0.0', 'sentencepiece>=0.1.99', 'tqdm>=4.66.0', 'accelerate>=0.30.0', 'peft>=0.11.0', 'bitsandbytes>=0.43.0', 'sacrebleu>=2.4.0']
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *packages], check=True)

In [ ]:
import shutil
def find_input(slug):
    for path in (Path('/kaggle/input') / slug, Path('/kaggle/input/datasets/attabeezy') / slug):
        if path.is_dir(): return path
    return None
source = find_input('akan-bpe-revision-v2-data')
if source is None: raise FileNotFoundError('Attach akan-bpe-revision-v2-data')
Path('data').mkdir(exist_ok=True)
for name in ('pristine_twi_train.jsonl', 'pristine_twi_test.jsonl'):
    shutil.copy(source / name, Path('data') / name)
prior = find_input('akan-bpe-downstream-results')
result_dir = Path('results/revision_v2/downstream/afrisenti/runs')
result_dir.mkdir(parents=True, exist_ok=True)
if prior:
    for path in prior.glob('*.json'): shutil.copy(path, result_dir / path.name)

In [ ]:
subprocess.run([sys.executable, 'scripts/run_downstream_afrisenti.py', 'validate'], check=True)
subprocess.run([sys.executable, 'scripts/run_downstream_afrisenti.py', 'fetch-data'], check=True)
subprocess.run([sys.executable, 'scripts/run_downstream_afrisenti.py', 'status'], check=True)

In [ ]:
import json, time
start = time.monotonic()
budget = 8 * 3600
reserve = 100 * 60
while budget - (time.monotonic() - start) >= reserve:
    status = json.loads(subprocess.run([sys.executable, 'scripts/run_downstream_afrisenti.py', 'status'], capture_output=True, text=True, check=True).stdout)
    if not status['pending']: break
    subprocess.run([sys.executable, 'scripts/run_downstream_afrisenti.py', 'run', '--next'], check=True)
subprocess.run([sys.executable, 'scripts/run_downstream_afrisenti.py', 'status'], check=True)

In [ ]:
status = json.loads(subprocess.run([sys.executable, 'scripts/run_downstream_afrisenti.py', 'status'], capture_output=True, text=True, check=True).stdout)
if not status['pending'] and not status['invalid']:
    subprocess.run([sys.executable, 'scripts/run_downstream_afrisenti.py', 'aggregate'], check=True)
Path('outputs').mkdir(exist_ok=True)
subprocess.run(['zip', '-r', '-j', 'outputs/afrisenti_downstream_results.zip', str(result_dir)], check=True)
print(status)